<a href="https://colab.research.google.com/github/compudiego/Nonlocality-Inequalities-in-Quantum-Computers-installable/blob/main/Nonlocality_Inequalities_in_Quantum_Computers_spinq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clone the project repository to access code and history
!git clone https://github.com/compudiego/Nonlocality-Inequalities-in-Quantum-Computers.git

# Set the working directory to the project root folder
%cd Nonlocality-Inequalities-in-Quantum-Computers

Cloning into 'Nonlocality-Inequalities-in-Quantum-Computers'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 56 (delta 31), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 32.29 KiB | 10.76 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/Nonlocality-Inequalities-in-Quantum-Computers/Nonlocality-Inequalities-in-Quantum-Computers/Nonlocality-Inequalities-in-Quantum-Computers/Nonlocality-Inequalities-in-Quantum-Computers/Nonlocality-Inequalities-in-Quantum-Computers


In [ ]:
%%bash
# Check if spinqit is already installed to avoid redundant steps
if python3.10 -c "import spinqit" &>/dev/null; then
    echo "Python 3.10 and SpinQit environment is already set up. Skipping installation."
else
    echo "Configuring Python 3.10 and SpinQit environment..."

    # 1. Update packages and install python3.10 in silent mode (-qq)
    apt-get update -qq
    apt-get install -qq python3.10 python3.10-distutils -y &>/dev/null

    # 2. Download and install pip only if not present
    if ! python3.10 -m pip --version &>/dev/null; then
        curl -s https://bootstrap.pypa.io/get-pip.py -o get-pip.py
        python3.10 get-pip.py --quiet
    fi

    # 3. Install required dependencies silently in the background
    python3.10 -m pip install --quiet spinqit scipy matplotlib
    python3.10 -m pip install --quiet --upgrade --ignore-installed numpy==1.26.4

    echo "Environment installed successfully!"
fi

# Sanity check to confirm spinqit imports properly
python3.10 -c "import os; os.environ['MPLBACKEND'] = 'Agg'; import spinqit; print('SpinQit and Python 3.10 ready.')"

Configuring Python 3.10 and SpinQit environment...
Environment installed successfully!
SpinQit and Python 3.10 ready.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%writefile circuits_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import math as m
import random
import numpy as np

from spinqit import Circuit, H, CX, Rx, Ry, Rz, CZ, SWAP
from spinqit import get_compiler, get_basic_simulator, BasicSimulatorConfig

def ghz_circuit_spinq(N=3):
    """Generates an N-qubit GHZ state circuit and returns (circ, q)."""
    circ = Circuit()
    q = circ.allocateQubits(N)
    circ << (H, q[0])
    for j in range(N - 1):
        circ << (CX, (q[j], q[j+1]))
    return circ, q

def QRC_spinq(N=3, D=5, max_gates=2):
    """Generates a Quantum Random Circuit (QRC) and returns (circ, q)."""
    Gate_Names_1q = ['H', 'Rx', 'Ry', 'Rz']
    Gate_Names_2q = ['CX', 'CZ', 'SWAP']

    circ = Circuit()
    q = circ.allocateQubits(N)

    for _ in range(D):
        qubits_numbers = list(range(N))
        choices = []

        while len(qubits_numbers) > 1:
            a = random.randint(1, min(max_gates, len(qubits_numbers)))
            b = random.sample(qubits_numbers, k=a)
            choices.append(b)
            for x in b:
                qubits_numbers.remove(x)

        if len(qubits_numbers) > 0:
            choices.append([qubits_numbers[0]])

        for x in choices:
            if len(x) == 1:
                gate_name = random.choice(Gate_Names_1q)
                target = q[x[0]]
                if gate_name == 'H':
                    circ << (H, target)
                elif gate_name == 'Rx':
                    circ << (Rx, target, 2 * m.pi * random.random())
                elif gate_name == 'Ry':
                    circ << (Ry, target, 2 * m.pi * random.random())
                elif gate_name == 'Rz':
                    circ << (Rz, target, 2 * m.pi * random.random())

            elif len(x) == 2:
                gate_name = random.choice(Gate_Names_2q)
                target = (q[x[0]], q[x[1]])
                if gate_name == 'CX':
                    circ << (CX, target)
                elif gate_name == 'CZ':
                    circ << (CZ, target)
                elif gate_name == 'SWAP':
                    circ << (SWAP, target)

    return circ, q

print("circuits_spinq.py file updated successfully.")

Writing circuits_spinq.py


In [ ]:
%%writefile test_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from circuits_spinq import ghz_circuit_spinq
from spinqit import get_compiler, get_basic_simulator, BasicSimulatorConfig

# Unpack the tuple (circ, q) returned by ghz_circuit_spinq
ghz, q = ghz_circuit_spinq(N=3)

# Compile and execute on the local SpinQit simulator
comp = get_compiler('native')
exe = comp.compile(ghz, 0)
engine = get_basic_simulator()

config = BasicSimulatorConfig()
config.configure_shots(1000)
res = engine.execute(exe, config)

print("GHZ test results on SpinQit simulator:")
print(res.counts)

Writing test_spinq.py


In [ ]:
!python3.10 test_spinq.py

circuits_spinq.py file updated successfully.
GHZ test results on SpinQit simulator:
{'000': 500, '111': 500}


In [ ]:
%%writefile observables_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import math as m
import numpy as np
from spinqit import Circuit, H, Rx, Ry

def append_pauli_measurement(circuit, qubits, bases):
    """
    Applies basis rotation gates to 'circuit' on the given 'qubits'
    according to the specified measurement bases ('X', 'Y', 'Z').
    'bases' is a string or list of length N, e.g., ['X', 'Z', 'Y']
    """
    for i, base in enumerate(bases):
        base = base.upper()
        if base == 'X':
            # Rotation to X basis
            circuit << (Ry, qubits[i], -m.pi / 2)
        elif base == 'Y':
            # Rotation to Y basis
            circuit << (Rx, qubits[i], m.pi / 2)
        elif base == 'Z':
            # Z basis is the natural measurement basis
            pass
        else:
            raise ValueError(f"Unsupported measurement basis: {base}")

    return circuit

def expectation_value_from_counts(counts, shots):
    """
    Calculates the parity expectation value <A_1 A_2 ... A_N> by assigning
    +1 to bitstrings with an even number of 1s and -1 to odd ones.
    """
    exp_val = 0.0
    for bitstring, count in counts.items():
        ones = bitstring.count('1')
        sign = 1 if ones % 2 == 0 else -1
        exp_val += sign * (count / shots)

    return exp_val

print("observables_spinq.py module updated successfully.")

Writing observables_spinq.py


In [ ]:
%%writefile test_observables.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import math as m
from spinqit import Circuit, H, CX, get_compiler, get_basic_simulator, BasicSimulatorConfig
from observables_spinq import append_pauli_measurement, expectation_value_from_counts

# 1. Create a 3-qubit GHZ circuit and store qubit references
N = 3
circ = Circuit()
q = circ.allocateQubits(N)

circ << (H, q[0])
for j in range(N - 1):
    circ << (CX, (q[j], q[j+1]))

# 2. Apply rotations to measure in the XXX basis
circ_xxx = append_pauli_measurement(circ, q, ['X', 'X', 'X'])

# 3. Compile and simulate using SpinQit
comp = get_compiler('native')
exe = comp.compile(circ_xxx, 0)
engine = get_basic_simulator()

shots = 2000
config = BasicSimulatorConfig()
config.configure_shots(shots)
res = engine.execute(exe, config)

# 4. Calculate parity expectation value <XXX>
val_esperado = expectation_value_from_counts(res.counts, shots)

print("Counts obtained in XXX basis:", res.counts)
print(f"Expectation value <XXX>: {val_esperado:.4f}")

Writing test_observables.py


In [ ]:
!python3.10 test_observables.py

observables_spinq.py module updated successfully.
Counts obtained in XXX basis: {'000': 500, '011': 500, '101': 500, '110': 500}
Expectation value <XXX>: 1.0000


In [ ]:
%%writefile inequalities_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from circuits_spinq import ghz_circuit_spinq
from observables_spinq import append_pauli_measurement, expectation_value_from_counts
from spinqit import Circuit, H, CX, get_compiler, get_basic_simulator, BasicSimulatorConfig

def measure_basis_expectation(circuit_fn, qubits_fn, bases, shots=2000):
    """
    Executes the circuit with basis rotations specified in 'bases'
    and returns the parity expectation value.
    """
    # Create a fresh copy of the circuit for measurement
    circ, q = circuit_fn()
    circ_rot = append_pauli_measurement(circ, q, bases)

    comp = get_compiler('native')
    exe = comp.compile(circ_rot, 0)
    engine = get_basic_simulator()

    config = BasicSimulatorConfig()
    config.configure_shots(shots)
    res = engine.execute(exe, config)

    return expectation_value_from_counts(res.counts, shots)

def evaluate_mabk_ghz(shots=2000):
    """
    Calculates the value of Mermin's M3 operator on a 3-qubit GHZ state.
    M3 = <XXX> - <XYY> - <YXY> - <YYX>
    """
    def build_ghz():
        circ = Circuit()
        q = circ.allocateQubits(3)
        circ << (H, q[0])
        circ << (CX, (q[0], q[1]))
        circ << (CX, (q[1], q[2]))
        return circ, q

    # Measure the 4 terms of the MABK inequality
    exp_xxx = measure_basis_expectation(build_ghz, None, ['X', 'X', 'X'], shots)
    exp_xyy = measure_basis_expectation(build_ghz, None, ['X', 'Y', 'Y'], shots)
    exp_yxy = measure_basis_expectation(build_ghz, None, ['Y', 'X', 'Y'], shots)
    exp_yyx = measure_basis_expectation(build_ghz, None, ['Y', 'Y', 'X'], shots)

    m3_value = exp_xxx - exp_xyy - exp_yxy - exp_yyx

    terms = {
        '<XXX>': exp_xxx,
        '<XYY>': exp_xyy,
        '<YXY>': exp_yxy,
        '<YYX>': exp_yyx
    }

    return m3_value, terms

print("inequalities_spinq.py module created successfully.")

Writing inequalities_spinq.py


In [ ]:
%%writefile test_mabk.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from inequalities_spinq import evaluate_mabk_ghz

print("Calculating MABK inequality on SpinQit simulator...")
m3, terminos = evaluate_mabk_ghz(shots=3000)

print("\n--- Observable Results ---")
for term, val in terminos.items():
    print(f"  {term}: {val:.4f}")

print("\n------------------------------")
print(f"M3 Value Obtained: {m3:.4f}")
print("Classical Bound: <= 2.0")
print("Maximum Quantum Bound: 4.0")

if m3 > 2.0:
    print("\nNON-LOCALITY VIOLATION CONFIRMED!")
else:
    print("\nClassical bound was not exceeded.")

Writing test_mabk.py


In [ ]:
!python3.10 test_mabk.py

circuits_spinq.py file updated successfully.
observables_spinq.py module updated successfully.
inequalities_spinq.py module created successfully.
Calculating MABK inequality on SpinQit simulator...

--- Observable Results ---
  <XXX>: 1.0000
  <XYY>: -1.0000
  <YXY>: -1.0000
  <YYX>: -1.0000

------------------------------
M3 Value Obtained: 4.0000
Classical Bound: <= 2.0
Maximum Quantum Bound: 4.0

NON-LOCALITY VIOLATION CONFIRMED!


In [ ]:
%%writefile optimizer_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import math as m
import numpy as np
from scipy.optimize import minimize

from observables_spinq import expectation_value_from_counts
from spinqit import Circuit, Ry, Rz, get_compiler, get_basic_simulator, BasicSimulatorConfig

def append_parametric_measurement(circuit, qubits, params):
    """
    Applies parametric rotations Ry(theta) and Rz(phi) to each qubit before Z-basis measurement.
    params is a 6-element array: [theta1, phi1, theta2, phi2, theta3, phi3]
    """
    for i in range(len(qubits)):
        theta = params[2 * i]
        phi = params[2 * i + 1]
        circuit << (Rz, qubits[i], phi)
        circuit << (Ry, qubits[i], theta)
    return circuit

def measure_mabk_with_params(state_builder_fn, params_A, params_B, shots=1000):
    """
    Calculates the MABK operator for two measurement settings A and B per qubit:
    M3 = <A1 A2 A3> - <A1 B2 B3> - <B1 A2 B3> - <B1 B2 A3>
    """
    def get_expectation(params_setting):
        circ, q = state_builder_fn()
        circ_rot = append_parametric_measurement(circ, q, params_setting)

        comp = get_compiler('native')
        exe = comp.compile(circ_rot, 0)
        engine = get_basic_simulator()

        config = BasicSimulatorConfig()
        config.configure_shots(shots)
        res = engine.execute(exe, config)

        return expectation_value_from_counts(res.counts, shots)

    # Construct the 4 measurement setting combinations for the 3 qubits
    # params_A and params_B contain [theta, phi] for each qubit
    setting_AAA = [params_A[0], params_A[1], params_A[2], params_A[3], params_A[4], params_A[5]]
    setting_ABB = [params_A[0], params_A[1], params_B[2], params_B[3], params_B[4], params_B[5]]
    setting_BAB = [params_B[0], params_B[1], params_A[2], params_A[3], params_B[4], params_B[5]]
    setting_BBA = [params_B[0], params_B[1], params_B[2], params_B[3], params_A[4], params_A[5]]

    exp_AAA = get_expectation(setting_AAA)
    exp_ABB = get_expectation(setting_ABB)
    exp_BAB = get_expectation(setting_BAB)
    exp_BBA = get_expectation(setting_BBA)

    m3_value = exp_AAA - exp_ABB - exp_BAB - exp_BBA
    return m3_value

def optimize_mabk_for_state(state_builder_fn, shots=1000, maxiter=50):
    """
    Optimizes measurement angles (12 parameters total: 6 for A and 6 for B)
    using SciPy COBYLA to maximize the violation |M3|.
    """
    # 12 initial random parameters between 0 and 2*pi
    x0 = np.random.uniform(0, 2 * np.pi, 12)

    def objective(x):
        params_A = x[:6]
        params_B = x[6:]
        # Minimizing negative value corresponds to maximizing M3
        m3 = measure_mabk_with_params(state_builder_fn, params_A, params_B, shots=shots)
        return -abs(m3)

    print("Starting SciPy optimization (COBYLA)...")
    res = minimize(objective, x0, method='COBYLA', options={'maxiter': maxiter})

    max_m3 = -res.fun
    return max_m3, res.x

print("optimizer_spinq.py module created successfully.")

Writing optimizer_spinq.py


In [ ]:
%%writefile test_optimizer.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from circuits_spinq import QRC_spinq
from optimizer_spinq import optimize_mabk_for_state

# Define builder function calling QRC_spinq directly
def build_qrc_state():
    return QRC_spinq(N=3, D=3, max_gates=2)

print("--- Testing SciPy optimization on a Quantum Random Circuit (QRC) ---")
m3_opt, best_params = optimize_mabk_for_state(build_qrc_state, shots=1000, maxiter=20)

print(f"\nOptimized M3 value reached: {m3_opt:.4f}")
print("Classical bound: <= 2.0")

if m3_opt > 2.0:
    print("SciPy algorithm found a measurement setting that violates classical localism for this QRC!")
else:
    print("The generated random state did not show non-local violation in this iteration.")

Writing test_optimizer.py


In [ ]:
!python3.10 test_optimizer.py

circuits_spinq.py file updated successfully.
observables_spinq.py module updated successfully.
optimizer_spinq.py module created successfully.
--- Testing SciPy optimization on a Quantum Random Circuit (QRC) ---
Starting SciPy optimization (COBYLA)...

Optimized M3 value reached: 1.1220
Classical bound: <= 2.0
The generated random state did not show non-local violation in this iteration.


In [ ]:
%%writefile backend_spinq.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from spinqit import get_compiler, get_basic_simulator, BasicSimulatorConfig

def get_spinq_execution_engine(mode='simulator', ip=None, task_name='mabk_task', shots=1000):
    """
    Backend switch to alternate between local simulator and SpinQ Triangulum II hardware.

    Parameters:
        mode (str): 'simulator' for local simulation or 'hardware' for SpinQ NMR.
        ip (str): IP address of the SpinQ Triangulum II on the local lab network.
        task_name (str): Task identifier for the compiler/hardware execution.
        shots (int): Number of execution shots.
    """
    comp = get_compiler('native')

    if mode == 'simulator':
        engine = get_basic_simulator()
        config = BasicSimulatorConfig()
        config.configure_shots(shots)
        return comp, engine, config

    elif mode == 'hardware':
        if not ip:
            raise ValueError("To execute on physical hardware, you must specify the SpinQ IP address (e.g., ip='192.168.1.15').")

        try:
            # Attempt to load SpinQit hardware connectors
            from spinqit.backend import NMR
            from spinqit import NMRConfig

            engine = NMR()
            config = NMRConfig()
            config.configure_ip(ip)
            config.configure_shots(shots)
            config.configure_task_name(task_name)
            return comp, engine, config

        except (ImportError, ModuleNotFoundError):
            raise RuntimeError(
                f"SpinQ NMR hardware connectors were not detected in this spinqit version.\n"
                f"When executing at the UNAHUR laboratory connected to the SpinQ device,\n"
                f"ensure the full SpinQ SDK is installed or verify the IP address ({ip})."
            )
    else:
        raise ValueError(f"Unrecognized execution mode: {mode}. Use 'simulator' or 'hardware'.")

print("backend_spinq.py module updated and ready for simulation and lab hardware.")

Writing backend_spinq.py


In [ ]:
%%writefile test_backend.py
import os
os.environ['MPLBACKEND'] = 'Agg'

from circuits_spinq import ghz_circuit_spinq
from backend_spinq import get_spinq_execution_engine

# 1. Test execution using Simulator Mode
print("--- Testing execution via Switch (Mode: Simulator) ---")
comp, engine, config = get_spinq_execution_engine(mode='simulator', shots=1000)

circ, q = ghz_circuit_spinq(N=3)
exe = comp.compile(circ, 0)
res = engine.execute(exe, config)

print("Simulator Counts:", res.counts)

# 2. Validate configuration structure for Hardware Mode (without active physical connection)
print("\n--- Validating configuration for Hardware Mode (SpinQ NMR) ---")
try:
    # Replace with real local IP of the SpinQ when inside the lab (e.g., '192.168.1.100')
    comp_hw, engine_hw, config_hw = get_spinq_execution_engine(mode='hardware', ip='127.0.0.1', shots=1000)
    print("Switch ready to connect to NMR hardware.")
except Exception as e:
    print(f"Hardware mode captured correctly (no physical device attached): {e}")

Writing test_backend.py


In [ ]:
!python3.10 test_backend.py

circuits_spinq.py file updated successfully.
backend_spinq.py module updated and ready for simulation and lab hardware.
--- Testing execution via Switch (Mode: Simulator) ---
Simulator Counts: {'000': 500, '111': 500}

--- Validating configuration for Hardware Mode (SpinQ NMR) ---
Hardware mode captured correctly (no physical device attached): SpinQ NMR hardware connectors were not detected in this spinqit version.
When executing at the UNAHUR laboratory connected to the SpinQ device,
ensure the full SpinQ SDK is installed or verify the IP address (127.0.0.1).
